# Sesión 4 — Del hallazgo a la decisión: el SCQA y el gráfico principal

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 4 de 9  
**Tipo:** Notebook guiado (caso del docente: Golden State Realty Advisors)

Hasta la sesión 3 el trabajo fue exploratorio: mirar los datos para entenderlos. La sesión 3 dejó los datos
revisados y una lista de «gangas». Hoy cambia el destinatario: todo es para un comité que decide. Este notebook
reproduce, paso a paso, cada cifra y cada gráfico de la presentación de la sesión.

## Agenda de la sesión

| Parte | Contenido |
|---|---|
| Quiz | Quiz de la sesión 3 |
| Repaso | Los datos revisados de la S3 |
| Parte 1 | El hallazgo candidato y la idea central |
| Parte 2 | Verificar contra comparables |
| Parte 3 | Cuánto se gana: potencial y escenarios |
| Parte 4 | Del gráfico exploratorio al gráfico principal |
| Parte 5 | El SCQA y la lista de verificación |

## Objetivos de aprendizaje

1. Distinguir el análisis exploratorio del explicativo e identificar el destinatario, la decisión y la idea central (Knaflic, 2015, pp. 19–31).
2. Verificar un hallazgo comparándolo con casos comparables (Diez et al., 2019, §1.3.4, p. 28).
3. Escribir el hallazgo como SCQA desde la mirada del comité y revisarlo con la lista de verificación (StrategyPunk, 2026, pp. 6–7; Knaflic, 2015, p. 172; la lista es criterio del curso).
4. Rediseñar un gráfico exploratorio como gráfico de decisión: tinta proporcional, desorden, atención y título de acción (Wilke, 2019, pp. 207–210 y 267; Knaflic, 2015, pp. 50–53, 71–126 y 141).
5. Expresar la oportunidad como ganancia sobre lo invertido, con el potencial como techo y un escenario declarado (definición del curso; Wilke, 2019, pp. 181–185).

## Problemática de negocio

**Golden State Realty Advisors** asesora a un fondo de inversionistas en vivienda de California. Su premisa:
comprar donde la vivienda se vende por debajo de lo que el ingreso de sus residentes sostiene.

- **Quién decide:** el comité de inversión, cada trimestre (perfil financiero, no técnico).
- **Qué decide hoy:** en qué distritos invierte el fondo este trimestre.
- **Restricción:** el capital del trimestre alcanza para invertir en unos **150 distritos** (supuesto del caso).
- **Antes de comprar:** el equipo hace la **debida diligencia** de cada distrito: la revisión a fondo (visita el
  distrito, revisa ventas recientes, estado de las viviendas, títulos y planes de la ciudad) que establece si el precio
  bajo es una oportunidad o tiene una razón que el censo no ve.
- **Qué está en juego:** si la lista es mala, el fondo pone su capital en distritos que el mercado ya valora bien y
  deja de invertir donde sí había oportunidad.

Licencia del caso: los datos son del censo de 1990 (Pace y Barry, 1997) y se tratan como actuales; se discute en la S8.

## Preparación del entorno

In [ ]:
import pathlib
import textwrap

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

In [ ]:
# Datos: censo de California de 1990 (Pace y Barry, 1997). Una fila por distrito censal.
FUENTE_GS = "Fuente: California Housing, censo de 1990 (Pace y Barry, 1997). Cálculos del curso."
crudo = pd.read_csv(DATOS / "housing.csv")
crudo["personas_hogar"] = crudo["population"] / crudo["households"]
# Indicador de la S2: cuántos años de ingreso cuesta la vivienda (median_income viene en decenas de miles de USD).
crudo["razon_precio_ingreso"] = crudo["median_house_value"] / (crudo["median_income"] * 10_000)
ZONAS = {"INLAND": "Interior", "<1H OCEAN": "A menos de 1 h del mar", "NEAR OCEAN": "Cerca del mar",
         "NEAR BAY": "Cerca de la bahía", "ISLAND": "Isla"}
CUANTIL = 0.05            # «el 5 % más barato»
CAPACIDAD = 150           # distritos en los que alcanza a invertir el capital del trimestre (supuesto del caso)
print(f"{len(crudo):,} distritos; razón precio/ingreso mediana: {crudo['razon_precio_ingreso'].median():.2f}")

## Repaso de la sesión anterior

Antes de seguir, el recorrido completo de Golden State. En la **S1** fijamos la pregunta del comité y su restricción
(150 distritos por trimestre). En la **S2** definimos el indicador, la razón precio/ingreso, y llamamos «ganga» a
todo distrito en el 5 % más bajo del estado: el primer barrido dio 1 032. En la **S3** revisamos los datos con seis
reglas de exclusión (**se limpia lo que la pregunta necesita**) y recalculamos el 5 %. La celda siguiente cuenta qué
habría llevado al comité un equipo que se saltara la S3; la figura resume el recorrido en una sola vista.

In [ ]:
reglas = {
    "Valor truncado abajo (14 999)": crudo["median_house_value"] <= 14_999,
    "Ingreso truncado (15)": crudo["median_income"] >= 15,
    "Más de 10 personas por hogar": crudo["personas_hogar"] > 10,
    "Menos de 100 habitantes": crudo["population"] < 100,
    "Valor truncado arriba (500 001)": crudo["median_house_value"] >= 500_001,
    "Zona ISLAND (5 distritos)": crudo["ocean_proximity"] == "ISLAND",
}
cand_crudo = crudo[crudo["razon_precio_ingreso"] <= crudo["razon_precio_ingreso"].quantile(CUANTIL)]
problemas = pd.Series({nombre: int(m.loc[cand_crudo.index].sum()) for nombre, m in reglas.items()})
print(f"Candidatos con los datos crudos: {len(cand_crudo):,}; con algún problema de datos: "
      f"{int(pd.concat([m.loc[cand_crudo.index] for m in reglas.values()], axis=1).any(axis=1).sum())}")
display(problemas.rename("candidatos afectados").to_frame())
display(cand_crudo.nsmallest(2, "razon_precio_ingreso")[["population", "median_income", "median_house_value",
                                                         "razon_precio_ingreso"]])

In [ ]:
# Figura: el recorrido de S1 a S3 en una sola vista: de dónde salen las «gangas» y los candidatos de hoy.
import textwrap
from matplotlib.patches import FancyBboxPatch

# Cifras del recorrido S1–S3 (todas se calculan aquí).
excluir = pd.concat(reglas.values(), axis=1).any(axis=1)
revisados = crudo[~excluir]
umbral_crudo = crudo["razon_precio_ingreso"].quantile(CUANTIL)
umbral_rev = revisados["razon_precio_ingreso"].quantile(CUANTIL)
cand_rev = revisados[revisados["razon_precio_ingreso"] <= umbral_rev]
con_problema = excluir.loc[cand_crudo.index]
n_problema = int(con_problema.sum())
n_dos_reglas = int((pd.concat(reglas.values(), axis=1).loc[cand_crudo.index].sum(axis=1) > 1).sum())
n_umbral = len(cand_crudo) - n_problema - len(cand_rev)
pct_int_crudo = 100 * (cand_crudo["ocean_proximity"] == "INLAND").mean()
pct_int_rev = 100 * (cand_rev["ocean_proximity"] == "INLAND").mean()
med_zona = revisados.groupby("ocean_proximity")["razon_precio_ingreso"].median()
costa = med_zona.drop("INLAND")
print(f"{miles(len(cand_crudo))} «gangas» crudas − {n_problema} con problemas de datos − {n_umbral} por el umbral "
      f"recalculado = {miles(len(cand_rev))} candidatos")

d = problemas  # candidatos afectados por cada regla (celda anterior)
tarjetas = [
    ("S1 · LA PREGUNTA", miles(CAPACIDAD), "distritos en los que alcanza a invertir el fondo", [
        "¿En qué distritos invertir? Donde la vivienda se vende por debajo de lo que el ingreso de sus residentes sostiene.",
        "El informe anterior llevó al comité a distritos que solo eran baratos por su ubicación."]),
    ("S2 · EL INDICADOR", miles(len(cand_crudo)), "«gangas» en el primer barrido", [
        "Razón precio/ingreso = valor mediano ÷ ingreso mediano: los años de ingreso que cuesta la vivienda "
        f"(mediana del estado: {decimal(crudo['razon_precio_ingreso'].median(), 2)}).",
        f"«Ganga»: el 5 % más bajo del estado (razón ≤ {decimal(umbral_crudo, 2)}).",
        f"El {pct_int_crudo:.0f} % está en el interior."]),
    ("S3 · LA LIMPIEZA", f"−{n_problema + n_umbral}", "«gangas» que salen", [
        f"Seis reglas excluyen {miles(excluir.sum())} de {miles(len(crudo))} distritos.",
        f"{n_problema} «gangas» eran problemas de datos: {d['Menos de 100 habitantes']} bloques con menos de 100 "
        f"habitantes, {d['Más de 10 personas por hogar']} hogares imposibles, {d['Valor truncado abajo (14 999)']} en el "
        f"piso de USD 14 999 (las dos mejores) y {d['Ingreso truncado (15)']} con el ingreso en el tope "
        f"({n_dos_reglas} caen en dos reglas).",
        f"Con los {miles(len(revisados))} distritos revisados se recalcula el 5 % y salen {n_umbral} más."]),
    ("HOY · LO QUE QUEDA ABIERTO", miles(len(cand_rev)), "candidatos con datos revisados", [
        f"{decimal(len(cand_rev) / CAPACIDAD, 1)} veces lo que alcanza a invertir el fondo.",
        f"El {pct_int_rev:.0f} % está en el interior, la zona más barata (razón mediana "
        f"{decimal(med_zona['INLAND'], 2)} frente a {decimal(costa.min(), 2)}–{decimal(costa.max(), 2)} en la costa).",
        "¿Son oportunidades o zonas que siempre son baratas?"]),
]

fig = plt.figure(figsize=(12, 5.9))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ancho, sep, x0 = 0.235, 0.012, 0.01
for i, (etiqueta, cifra, rotulo, lineas) in enumerate(tarjetas):
    x = x0 + i * (ancho + sep)
    hoy = i == 3
    ax.add_patch(FancyBboxPatch((x, 0.31), ancho, 0.67, boxstyle="round,pad=0,rounding_size=0.012",
                                facecolor=PALETA["fondo"], edgecolor=PALETA["principal"] if hoy else "none",
                                linewidth=1.5))
    ax.text(x + 0.015, 0.94, etiqueta, fontsize=10.5, fontweight="bold",
            color=PALETA["principal"] if hoy else PALETA["acento"], va="top")
    ax.text(x + 0.015, 0.875, cifra, fontsize=26, fontweight="bold",
            color=PALETA["principal"] if hoy else PALETA["marino"], va="top")
    ax.text(x + 0.015, 0.775, rotulo, fontsize=10.5, color=PALETA["secundario"], va="top")
    yt = 0.72
    for linea in lineas:  # un párrafo por idea, con un poco de aire entre párrafos
        t = ax.text(x + 0.015, yt, "\n".join(textwrap.wrap(linea, 40)), fontsize=10.5, color=PALETA["texto"],
                    va="top", linespacing=1.25)
        yt -= t.get_window_extent(fig.canvas.get_renderer()).height / fig.bbox.height + 0.022
    if i < 3:
        ax.annotate("", xy=(x + ancho + sep, 0.67), xytext=(x + ancho, 0.67),
                    arrowprops=dict(arrowstyle="-|>", color=PALETA["secundario"], lw=1.2))

# Barra: en qué se convierten las 1 032 «gangas» del primer barrido.
izq, der, y, h = 0.01, 0.99, 0.10, 0.075
escala = (der - izq) / len(cand_crudo)
partes = [(len(cand_rev), PALETA["principal"], f"{miles(len(cand_rev))} candidatos de hoy"),
          (n_umbral, PALETA["neutro"], f"{n_umbral} por el umbral recalculado"),
          (n_problema, PALETA["acento"], f"{n_problema} problemas de datos")]
inicio = izq
for n, color, _ in partes:
    ax.add_patch(plt.Rectangle((inicio, y), n * escala, h, facecolor=color, edgecolor="white", linewidth=0.8))
    inicio += n * escala
ax.text(izq + len(cand_rev) * escala / 2, y + h / 2, partes[0][2], fontsize=11, color="white",
        fontweight="bold", ha="center", va="center")
xt = der
for n, color, rot in partes[:0:-1]:
    t = ax.text(xt, y + h + 0.025, rot, fontsize=10.5, color=PALETA["texto"], ha="right", va="bottom")
    ancho_t = t.get_window_extent(fig.canvas.get_renderer()).width / fig.bbox.width
    ax.add_patch(plt.Rectangle((xt - ancho_t - 0.022, y + h + 0.028), 0.014, 0.03, facecolor=color))
    xt -= ancho_t + 0.045
ax.text(izq, y + h + 0.025, f"Las {miles(len(cand_crudo))} «gangas» del primer barrido", fontsize=10.5,
        fontweight="bold", color=PALETA["texto"], va="bottom")
xc = izq + CAPACIDAD * escala
ax.plot([xc, xc], [y - 0.025, y + h + 0.005], color=PALETA["marino"], linewidth=1.4, linestyle="--")
ax.text(xc, y - 0.035, f"Capital del trimestre: {CAPACIDAD} distritos", fontsize=10.5, color=PALETA["marino"],
        ha="center", va="top")

titulo(fig, f"De dónde salen los {miles(len(cand_rev))} candidatos: la pregunta, el indicador y la limpieza",
       "Golden State, sesiones 1 a 3. La barra descompone las «gangas» del primer barrido", y=1.06)
fuente(fig, FUENTE_GS)
guardar(fig, "recorrido_s1_s3.png")

In [ ]:
excluir = pd.concat(reglas.values(), axis=1).any(axis=1)
df = crudo[~excluir].copy()
df["nivel_ingreso"] = pd.qcut(df["median_income"], 3, labels=["Bajo", "Medio", "Alto"])
print(f"Distritos analizables: {len(df):,}")

## Parte 1 — El hallazgo candidato y la idea central

**Exploratorio frente a explicativo.** Explorar es buscar lo interesante; explicar es comunicar a otros una cosa
concreta que importa (Knaflic, 2015, pp. 19–20; Wilke, 2019, pp. 327–329). Antes de graficar: quién decide, qué
debe saber o hacer y la idea central en una oración (Knaflic, 2015, pp. 20–31).

**Hallazgo candidato:** un distrito es «ganga» si su razón precio/ingreso está en el 5 % más bajo de California.

In [ ]:
umbral_estado = df["razon_precio_ingreso"].quantile(CUANTIL)
df["candidato"] = df["razon_precio_ingreso"] <= umbral_estado
cand = df[df["candidato"]]
pct_interior = 100 * (cand["ocean_proximity"] == "INLAND").mean()
print(f"Umbral del 5 %: {umbral_estado:.2f}; candidatos: {len(cand):,}; en el interior: {pct_interior:.0f} %")

In [ ]:
# Figura: la distribución del indicador y el corte del 5 %.
fig, ax = plt.subplots(figsize=(9, 3.8))
bins = np.linspace(0, 12, 61)
ax.hist(df["razon_precio_ingreso"].clip(upper=12), bins=bins, color=PALETA["neutro"])
ax.hist(cand["razon_precio_ingreso"], bins=bins, color=PALETA["principal"])
ax.axvline(umbral_estado, color=PALETA["marino"], linewidth=1.2, linestyle="--")
ax.text(umbral_estado + 0.15, ax.get_ylim()[1] * 0.9, f"5 % más barato: razón ≤ {decimal(umbral_estado, 2)}\n"
        f"{miles(len(cand))} distritos, {pct_interior:.0f} % en el interior", fontsize=10.5, color=PALETA["texto"])
ax.set_xlabel("Razón precio/ingreso (años de ingreso que cuesta la vivienda)")
ax.set_ylabel("Distritos"); ax.set_xlim(0, 12)
titulo(fig, f"{miles(len(cand))} «gangas»: el 5 % de los distritos con la vivienda más barata para su ingreso",
       "Distribución de la razón precio/ingreso en California (datos revisados en la S3)")
fuente(fig, FUENTE_GS)
guardar(fig, "distribucion_razon.png")

In [ ]:
# Figura «antes»: la vista exploratoria del analista, con los valores por defecto.
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(9, 5))
    resto = df[~df["candidato"]]
    ax.scatter(resto["median_income"], resto["median_house_value"], s=4, alpha=0.35, color="#4C72B0", label="otros")
    ax.scatter(cand["median_income"], cand["median_house_value"], s=6, color="#DD8452", label="candidatos")
    ax.set_xlabel("median_income"); ax.set_ylabel("median_house_value")
    ax.set_title("median_income vs median_house_value")
    ax.legend(loc="upper left"); ax.grid(True, alpha=0.4)
    guardar(fig, "grafico_antes.png")

**Cómo leer la dispersión.** Cada punto es un distrito revisado. El eje horizontal es `median_income`, el ingreso
mediano del distrito en decenas de miles de USD (4 = USD 40 000); el vertical, `median_house_value`, el valor mediano
de la vivienda en USD. La razón precio/ingreso es la pendiente de la recta que une el origen con cada punto. Por eso el
5 % más bajo de la razón no es un rincón del gráfico: son los puntos que quedan **debajo de una recta** que sale del
origen con la pendiente del umbral. Los puntos naranja son esos candidatos.

In [ ]:
# Figura: la misma vista exploratoria, con la recta del 5 % que separa a los candidatos.
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(resto["median_income"], resto["median_house_value"], s=4, alpha=0.35, color="#4C72B0", label="otros")
    ax.scatter(cand["median_income"], cand["median_house_value"], s=6, color="#DD8452", label="candidatos")
    x_recta = np.array([0, df["median_income"].max()])
    ax.plot(x_recta, umbral_estado * 10_000 * x_recta, color="black", linewidth=1.2, linestyle="--")
    ax.annotate(f"Recta del 5 %: la vivienda cuesta {decimal(umbral_estado, 2)} años de ingreso.\n"
                f"Debajo quedan los {miles(len(cand))} candidatos",
                xy=(9, umbral_estado * 10_000 * 9), xytext=(4.6, 8_000), fontsize=10.5,
                arrowprops=dict(arrowstyle="->", color="black"), ha="left")
    ax.set_xlabel("median_income"); ax.set_ylabel("median_house_value")
    ax.set_title("median_income vs median_house_value")
    ax.legend(loc="upper left"); ax.grid(True, alpha=0.4)
    guardar(fig, "grafico_antes_guia.png")

### Punto de control 1 — La idea central

**¿Para quién es este análisis y qué debe decidir?**
Para el comité de inversión: en qué distritos invierte el fondo este trimestre, con capital para unos 150.

**¿Resiste la primera idea central («hay 971 distritos baratos para su ingreso»)?**
No: casi todos están en el interior, y el comité preguntará de inmediato si no es solo que el interior es más barato.

> ¿Qué haría el comité con el gráfico «antes»? ¿Qué tendría que descubrir por su cuenta?

## Parte 2 — Verificar contra comparables

**La sospecha.** Si las «gangas» se concentran donde todo es más barato, puede que no sean oportunidades: la zona
explica a la vez que el precio sea bajo y que el distrito entre en la lista. A esa tercera variable se le llama
**variable de confusión**: «correlacionada tanto con la variable explicativa como con la de respuesta» (Diez et al.,
2019, §1.3.4, p. 28).

**Cómo se controla.** Cada distrito se compara solo con sus parecidos: primero los de su misma zona y luego los de su
misma zona y su mismo nivel de ingreso. En el diseño de experimentos, agrupar así se llama *blocking* (Diez et al.,
2019, §1.4.1, p. 36; aquí se usa como analogía). Verificar no es probar: puede haber otras variables que no medimos.

**Regla:** un candidato se confirma si también está en el 5 % más barato **de su grupo de comparables**.

In [ ]:
# Figura: la sospecha. ¿Dónde están los distritos y dónde los candidatos?
orden_z = ["INLAND", "<1H OCEAN", "NEAR OCEAN", "NEAR BAY"]
pct_distritos = 100 * df["ocean_proximity"].value_counts(normalize=True).reindex(orden_z)
pct_candidatos = 100 * cand["ocean_proximity"].value_counts(normalize=True).reindex(orden_z).fillna(0)
razon_zona = df.groupby("ocean_proximity")["razon_precio_ingreso"].median().reindex(orden_z)
fig, ax = plt.subplots(figsize=(7.4, 4.1))
y = np.arange(len(orden_z))[::-1]
ax.barh(y + 0.19, pct_distritos, height=0.36, color=PALETA["neutro"], label="% de los distritos del estado")
ax.barh(y - 0.19, pct_candidatos, height=0.36, color=PALETA["principal"], label=f"% de los {miles(len(cand))} candidatos")
for yi, a, b in zip(y, pct_distritos, pct_candidatos):
    ax.text(a + 1, yi + 0.19, f"{a:.0f} %", va="center", fontsize=10.5, color=PALETA["texto"])
    ax.text(b + 1, yi - 0.19, f"{b:.0f} %", va="center", fontsize=10.5, color=PALETA["texto"], fontweight="bold")
ax.set_yticks(y, [ZONAS[z] for z in orden_z]); limpiar_ejes(ax); ax.set_xlim(0, 125)
for yi, z in zip(y, orden_z):
    ax.text(112, yi, decimal(razon_zona[z], 2), va="center", ha="center", fontsize=10.5, color=PALETA["secundario"])
ax.text(112, y[0] + 0.45, "Razón mediana", ha="center", va="bottom", fontsize=9.5, color=PALETA["secundario"])
ax.set_ylim(y[-1] - 0.6, y[0] + 0.8)
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(0.85, 0.0), fontsize=10)
titulo(fig, f"El interior tiene el {pct_distritos['INLAND']:.0f} % de los distritos y el {pct_candidatos['INLAND']:.0f} % de los candidatos",
       "Por zona: distritos revisados y candidatos; a la derecha, la razón mediana")
fuente(fig, FUENTE_GS)
guardar(fig, "zona_candidatos.png")

In [ ]:
def bajo_cuantil(datos, grupos, q=CUANTIL):
    umbral = datos.groupby(grupos, observed=True)["razon_precio_ingreso"].transform(lambda s: s.quantile(q))
    return datos["razon_precio_ingreso"] <= umbral


df["resiste_zona"] = df["candidato"] & bajo_cuantil(df, ["ocean_proximity"])
df["confirmado"] = df["resiste_zona"] & bajo_cuantil(df, ["ocean_proximity", "nivel_ingreso"])
embudo = pd.Series({"Candidatos sin revisar los datos": len(cand_crudo),
                    "Candidatos con datos revisados (S3)": int(df["candidato"].sum()),
                    "Resisten la comparación con su zona": int(df["resiste_zona"].sum()),
                    "Resisten zona e ingreso: confirmados": int(df["confirmado"].sum())})
display(embudo.to_frame("distritos"))
corte_zona = df.groupby("ocean_proximity")["razon_precio_ingreso"].quantile(CUANTIL)
corte_zona_ingreso = df.groupby(["ocean_proximity", "nivel_ingreso"], observed=True)["razon_precio_ingreso"].quantile(CUANTIL)
limites_ingreso = df.groupby("nivel_ingreso", observed=True)["median_income"].agg(["min", "max"]) * 10_000
display(corte_zona.rename(index=ZONAS).round(2).to_frame("corte del 5 % de la zona"))
display(limites_ingreso.round(-1).astype(int))

**Cómo leer la figura siguiente.** Cada fila es una zona y cada punto, un candidato, ubicado según su razón
precio/ingreso. La línea discontinua es el corte del estado (todos los candidatos quedan a su izquierda). La marca
verde es el corte de la zona: el candidato que queda a su izquierda resiste; el que queda entre las dos líneas solo
era barato frente al estado.

In [ ]:
# Figura: cada candidato frente al corte de su propia zona.
rng = np.random.default_rng(2026)
fig, ax = plt.subplots(figsize=(10, 3.9))
for fila, z in enumerate(orden_z[::-1]):
    d = df[df["candidato"] & (df["ocean_proximity"] == z)]
    yy = fila + rng.uniform(-0.28, 0.28, len(d))
    ax.scatter(d["razon_precio_ingreso"], yy, s=9, linewidths=0,
               color=np.where(d["resiste_zona"], PALETA["principal"], PALETA["neutro"]))
    ax.plot([corte_zona[z]] * 2, [fila - 0.38, fila + 0.38], color=PALETA["principal"], linewidth=2.5)
    ax.text(corte_zona[z], fila + 0.42, f"corte de la zona: {decimal(corte_zona[z], 2)}", ha="center", va="bottom",
            fontsize=9.5, color=PALETA["principal"])
    ax.text(3.85, fila, f"{miles(len(d))} candidatos, {miles(int(d['resiste_zona'].sum()))} resisten", va="center",
            fontsize=10.5, color=PALETA["texto"])
ax.axvline(umbral_estado, color=PALETA["secundario"], linestyle="--", linewidth=1.2)
ax.text(umbral_estado, -0.75, f"corte del estado: {decimal(umbral_estado, 2)}", ha="center", fontsize=9.5,
        color=PALETA["secundario"])
ax.set_yticks(range(len(orden_z)), [ZONAS[z] for z in orden_z[::-1]])
ax.set_xlim(0.6, 3.8); ax.set_ylim(-0.9, len(orden_z) - 0.3)
ax.set_xlabel("Razón precio/ingreso (años de ingreso que cuesta la vivienda)")
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
descartados = int((df["candidato"] & ~df["resiste_zona"]).sum())
cand_interior = int((df["candidato"] & (df["ocean_proximity"] == "INLAND")).sum())
titulo(fig, f"En el interior, {miles(descartados)} de los {miles(cand_interior)} candidatos son baratos solo frente al estado; "
       "en la costa resisten todos",
       "Cada punto es un candidato. Verde: también está en el 5 % más barato de su zona; gris: no", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "umbral_por_zona.png")

**El segundo paso: zona y nivel de ingreso.** El nivel de ingreso son los tercios del estado (bajo, medio y alto,
según el ingreso mediano del distrito; los límites están en la tabla de arriba). El comité preguntaría: ¿no será que
los distritos de cierto ingreso pagan siempre menos años de ingreso? Por eso se repite la comparación dentro de cada
zona y nivel de ingreso.

In [ ]:
# Los 12 cortes del paso 3: el 5 % más barato de cada grupo de zona y nivel de ingreso.
display(corte_zona_ingreso.unstack().rename(index=ZONAS).round(2))

In [ ]:
# Figura: el paso 3. Los que resistieron su zona, frente al corte de su grupo de zona e ingreso.
rng = np.random.default_rng(2026)
filas_p3 = [("Interior, ingreso bajo", (df["ocean_proximity"] == "INLAND") & (df["nivel_ingreso"] == "Bajo")),
            ("Interior, ingreso medio", (df["ocean_proximity"] == "INLAND") & (df["nivel_ingreso"] == "Medio")),
            ("Interior, ingreso alto", (df["ocean_proximity"] == "INLAND") & (df["nivel_ingreso"] == "Alto")),
            ("Costa (tres zonas)", df["ocean_proximity"] != "INLAND")]
fig, ax = plt.subplots(figsize=(10, 3.9))
for fila, (nombre, m) in enumerate(filas_p3[::-1]):
    d = df[df["resiste_zona"] & m]
    yy = fila + rng.uniform(-0.28, 0.28, len(d))
    ax.scatter(d["razon_precio_ingreso"], yy, s=9, linewidths=0,
               color=np.where(d["confirmado"], PALETA["principal"], PALETA["neutro"]))
    if nombre.startswith("Interior"):
        nivel = nombre.split()[-1].capitalize()
        corte = corte_zona_ingreso.loc[("INLAND", nivel)]
        ax.plot([corte] * 2, [fila - 0.38, fila + 0.38], color=PALETA["principal"], linewidth=2.5)
        ax.text(corte + (0.03 if corte > corte_zona["INLAND"] else -0.03), fila + 0.42, f"corte del grupo: {decimal(corte, 2)}",
                ha="left" if corte > corte_zona["INLAND"] else "right", va="bottom", fontsize=9.5, color=PALETA["principal"])
    else:
        cortes_costa = corte_zona_ingreso.drop("INLAND", level=0)
        ax.text(0.65, fila + 0.42, f"cortes de los grupos de la costa: {decimal(cortes_costa.min(), 2)} a "
                f"{decimal(cortes_costa.max(), 2)} (a la derecha de todos)", ha="left", va="bottom", fontsize=9.5,
                color=PALETA["principal"])
    ax.text(2.74, fila, f"{miles(int(d['confirmado'].sum()))} de {miles(len(d))} resisten", va="center", fontsize=10.5,
            color=PALETA["texto"])
ax.axvline(corte_zona["INLAND"], ymin=0.25, color=PALETA["secundario"], linestyle="--", linewidth=1.2)
ax.text(corte_zona["INLAND"], len(filas_p3) - 0.35, f"corte de la zona interior (paso 2): {decimal(corte_zona['INLAND'], 2)}",
        ha="center", va="bottom", fontsize=9.5, color=PALETA["secundario"])
ax.set_yticks(range(len(filas_p3)), [n for n, _ in filas_p3[::-1]])
ax.set_xlim(0.6, 3.3); ax.set_ylim(-0.6, len(filas_p3) + 0.05)
ax.set_xlabel("Razón precio/ingreso (años de ingreso que cuesta la vivienda)")
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
n_sale = int((df["resiste_zona"] & ~df["confirmado"]).sum())
titulo(fig, f"En el interior, {n_sale} candidatos de ingreso medio y alto no resisten a sus parecidos: quedan "
       f"{int(df['confirmado'].sum())}",
       f"Los {miles(int(df['resiste_zona'].sum()))} que resistieron su zona. Verde: también en el 5 % más barato de su "
       "zona y nivel de ingreso; gris: no", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "paso3_ingreso.png")

In [ ]:
# Figura: el embudo de la verificación, paso por paso, con quién sale y por qué.
n_cand, n_zona, n_conf = (int(df[c_].sum()) for c_ in ("candidato", "resiste_zona", "confirmado"))
sale_zona = df[df["candidato"] & ~df["resiste_zona"]]
sale_ingreso = df[df["resiste_zona"] & ~df["confirmado"]]
def donde(d):
    z = d["ocean_proximity"].map(ZONAS).value_counts()
    return "todos del interior" if list(z.index) == ["Interior"] else ", ".join(f"{v} {k.lower()}" for k, v in z.items())
niveles = sale_ingreso["nivel_ingreso"].astype(str).str.lower().value_counts().index.tolist()
c_int = corte_zona_ingreso.loc["INLAND"]
pasos = [
    ("1. Frente al estado", n_cand, f"Corte: {decimal(umbral_estado, 2)} años de ingreso", ""),
    ("2. Frente a su zona", n_zona, f"Salen {miles(len(sale_zona))}, {donde(sale_zona)}",
     "El interior es más barato en general"),
    ("3. Frente a su zona y\nsu nivel de ingreso", n_conf, f"Salen {miles(len(sale_ingreso))}, {donde(sale_ingreso)}, "
     f"de ingreso {' y '.join(niveles)}",
     f"En el interior, ese ingreso paga aún menos: corte {decimal(c_int['Medio'], 2)} "
     f"(frente a {decimal(c_int['Bajo'], 2)} en ingreso bajo)"),
    ("4. Los de mayor potencial", CAPACIDAD, "Se eligen en la Tarea 4: el capital del trimestre",
     f"alcanza para unos {CAPACIDAD} distritos"),
]
fig = plt.figure(figsize=(11, 4.6))
ax = fig.add_axes([0.2, 0.05, 0.3, 0.8])
y = np.arange(len(pasos))[::-1]
colores_emb = [PALETA["neutro"], PALETA["neutro"], PALETA["principal"], "#E3F1EF"]
ax.barh(y, [p_[1] for p_ in pasos], height=0.55, color=colores_emb,
        edgecolor=[c_ if c_ != "#E3F1EF" else PALETA["principal"] for c_ in colores_emb], linestyle="-")
for yi, (nombre, n, linea1, linea2) in zip(y, pasos):
    ax.text(n + 15, yi, miles(n), va="center", fontsize=13, fontweight="bold",
            color=PALETA["principal"] if yi in (0, 1) else PALETA["texto"])
    ax.annotate(f"{linea1}\n{linea2}".strip(), xy=(1.12, yi), xycoords=("axes fraction", "data"),
                va="center", fontsize=10.5, color=PALETA["texto"], linespacing=1.35, annotation_clip=False)
ax.set_yticks(y, [p_[0] for p_ in pasos], fontsize=11); limpiar_ejes(ax); ax.set_xlim(0, n_cand * 1.2)
titulo(fig, f"De {miles(n_cand)} candidatos quedan {n_conf}: cada comparación saca a los que solo eran baratos "
       "frente a un grupo más amplio", f"En la costa no sale ningún candidato. El paso 4 (los {CAPACIDAD} de mayor "
       "potencial) se hace en la Tarea 4, cuando se define el potencial", y=1.08)
fuente(fig, FUENTE_GS)
guardar(fig, "embudo.png")

In [ ]:
# Figura de respaldo: candidatos por zona, confirmados y descartados.
comp = (df[df["candidato"]].assign(estado=lambda d: np.where(d["confirmado"], "Confirmado", "Descartado"))
        .groupby("ocean_proximity")["estado"].value_counts().unstack(fill_value=0)
        .reindex(columns=["Confirmado", "Descartado"], fill_value=0))
comp["Candidatos"] = comp.sum(axis=1)
comp = comp.rename(index=ZONAS).sort_values("Candidatos", ascending=False)
display(comp)
fig, ax = plt.subplots(figsize=(9, 4))
y = list(range(len(comp)))[::-1]
ax.barh(y, comp["Confirmado"], color=PALETA["principal"], height=0.6)
ax.barh(y, comp["Descartado"], left=comp["Confirmado"], color=PALETA["neutro"], height=0.6, edgecolor="white", linewidth=2)
for yi, (z, f) in zip(y, comp.iterrows()):
    ax.text(f["Candidatos"] + 12, yi, f"{miles(f['Confirmado'])} de {miles(f['Candidatos'])}", va="center", fontsize=11)
interior = comp.loc["Interior"]
ax.text(interior["Confirmado"] + interior["Descartado"] / 2, y[0],
        f"{miles(interior['Descartado'])} descartados:\nbaratos solo por estar en el interior", va="center", ha="center", fontsize=10)
ax.text(interior["Confirmado"] / 2, y[0], "confirmados", va="center", ha="center", fontsize=10, color="white", fontweight="bold")
ax.set_yticks(y, comp.index); limpiar_ejes(ax); ax.set_xlim(0, comp["Candidatos"].max() * 1.18)
titulo(fig, f"Dos de cada tres «gangas» eran solo casas del interior: {int(df['confirmado'].sum())} distritos resisten",
       f"Candidatos ({miles(int(df['candidato'].sum()))}) por zona. Confirmado = también en el 5 % más barato frente a su zona e ingreso")
fuente(fig, FUENTE_GS)
guardar(fig, "grafico_respaldo_zona.png")

**Robustez.** El 5 % es una elección. Para ver si la lista depende de ella, se cambia por el 4 % (más estricto) y el
6 % (más amplio) en los tres pasos y se repite todo.

In [ ]:
# Robustez: ¿cambia la lista si el corte es 4 % o 6 %?
confirmados_q = {}
for q in (0.04, 0.05, 0.06):
    c_q = (df["razon_precio_ingreso"] <= df["razon_precio_ingreso"].quantile(q)) & bajo_cuantil(df, ["ocean_proximity"], q) \
        & bajo_cuantil(df, ["ocean_proximity", "nivel_ingreso"], q)
    confirmados_q[q] = set(df.index[c_q])
c4, c5, c6 = confirmados_q[0.04], confirmados_q[0.05], confirmados_q[0.06]
assert c4 <= c5 <= c6                                   # los conjuntos se contienen
tramos = [(len(c4), PALETA["principal"], "con cualquier corte"),
          (len(c5 - c4), PALETA["principal_claro"], "con el 5 %\ny el 6 %"),
          (len(c6 - c5), PALETA["neutro"], "solo con\nel 6 %")]
print({f"{int(q * 100)} %": len(s) for q, s in confirmados_q.items()})
# Qué cambia con cada corte: el valor de la razón en cada paso (ejemplo del interior).
interior_m = df["ocean_proximity"] == "INLAND"
cortes_q = pd.DataFrame({
    f"{int(q * 100)} %": {
        "Estado (candidatos)": f"{decimal(df['razon_precio_ingreso'].quantile(q), 2)} "
                               f"({miles((df['razon_precio_ingreso'] <= df['razon_precio_ingreso'].quantile(q)).sum())})",
        "Zona: interior": decimal(df.loc[interior_m, "razon_precio_ingreso"].quantile(q), 2),
        "Interior, ingreso medio": decimal(df.loc[interior_m & (df["nivel_ingreso"] == "Medio"), "razon_precio_ingreso"].quantile(q), 2),
    } for q in (0.04, 0.05, 0.06)})
display(cortes_q)
fig = plt.figure(figsize=(11.5, 3.8))
at = fig.add_axes([0.0, 0.2, 0.38, 0.45]); at.set_axis_off()
tabla_q = at.table(cellText=cortes_q.values, rowLabels=cortes_q.index, colLabels=cortes_q.columns,
                   cellLoc="center", loc="center", colWidths=[0.22] * 3)
tabla_q.auto_set_font_size(False); tabla_q.set_fontsize(12); tabla_q.scale(1, 2.0)
for (fila_t, col_t), celda in tabla_q.get_celld().items():
    celda.set_edgecolor("white")
    celda.set_facecolor(PALETA["marino"] if fila_t == 0 else (PALETA["fondo"] if fila_t % 2 else "white"))
    if fila_t == 0:
        celda.get_text().set_color("white"); celda.get_text().set_fontweight("bold")
at.set_title("Qué cambia con cada corte (razón precio/ingreso)", fontsize=12, loc="right", color=PALETA["texto"])
ax = fig.add_axes([0.47, 0.05, 0.53, 0.8])
inicio = 0
for n, color, rot in tramos:
    ax.barh(0, n, left=inicio, color=color, height=0.5, edgecolor="white", linewidth=1.5)
    if n > 0.3 * len(c6):
        ax.text(inicio + n / 2, 0, f"{n}\n{rot}", ha="center", va="center", fontsize=10.5, color="white")
    else:
        ax.text(inicio + n / 2, 0, str(n), ha="center", va="center", fontsize=11,
                color="white" if color != PALETA["neutro"] else PALETA["texto"])
        ax.text(inicio + n / 2, -0.33, rot, ha="center", va="top", fontsize=10, color=PALETA["texto"])
    inicio += n
for k, (n, rot) in enumerate(((len(c4), "4 %"), (len(c5), "5 %"), (len(c6), "6 %"))):
    yb = 0.38 + 0.2 * k
    ax.plot([0, 0, n, n], [yb - 0.05, yb, yb, yb - 0.05], color=PALETA["marino"], linewidth=1.1)
    ax.text(n + 4, yb, f"con el {rot}: {n}", va="center", fontsize=10.5, color=PALETA["marino"])
ax.set_xlim(0, len(c6) * 1.22); ax.set_ylim(-0.65, 1.05); ax.set_axis_off()
titulo(fig, f"Con un corte más estricto o más amplio, los {len(c5)} se mantienen: el 4 % es un subconjunto y el 6 % los contiene",
       "El 5 % es una elección: lo cambiamos por el 4 % (más estricto) y el 6 % (más amplio) en los tres pasos y "
       "repetimos todo", y=1.12)
fuente(fig, FUENTE_GS)
guardar(fig, "robustez.png")

### Punto de control 2 — Comparables

**¿Por qué ningún candidato costero se descarta?**
En la costa el precio es alto en general; un costero en el 5 % más bajo del estado es barato también frente a su zona.

**Si los descartados no son oportunidad, ¿qué son?**
Distritos que el mercado valora bien: baratos porque están en el interior, no porque estén subvalorados.

**¿Por qué no basta con comparar por zona?**
El comité preguntaría por el ingreso; la segunda comparación quita más candidatos.

> No hay garantía de medir todas las variables de confusión (Diez et al., 2019, p. 28). ¿Qué otra variable preguntaría el comité?

## Parte 3 — Cuánto vale actuar: potencial y escenarios

La pregunta del comité es financiera: **si invertimos USD 1 M en estos distritos, ¿cuánto podríamos ganar?** Para
responderla, «barato frente a sus comparables» se traduce a dólares en tres pasos (definición del curso):

1. **¿Cuánto debería valer?** Precio esperado = lo que pagan sus comparables por año de ingreso × su ingreso:
   $P_{\mathrm{esp}} = \tilde{r}_{\mathrm{comp}} \times I$.
2. **¿Qué tan por debajo está?** Potencial = precio esperado ÷ precio actual − 1: $v = P_{\mathrm{esp}}/P - 1$. Es un
   **techo**: supone que el precio alcanza por completo a sus comparables.
3. **¿Cuánto se gana con un supuesto prudente?** Si el precio sube una fracción $f$ de lo que le falta, cada dólar
   gana $f \times v$. Con USD 1 M repartido por igual entre los distritos: $G = 10^6 \times f \times \bar{v}$. Como $f$
   no se conoce, se muestran escenarios (Wilke, 2019, pp. 181–185): una cuarta parte y la mitad.

Se expresa por dólar invertido porque así compara el comité; sumar diferencias de precio entre distritos no es una
ganancia del fondo.

In [ ]:
ESCENARIOS = (0.25, 0.50)
mediana_grupo = df.groupby(["ocean_proximity", "nivel_ingreso"], observed=True)["razon_precio_ingreso"].transform("median")
df["precio_esperado"] = mediana_grupo * df["median_income"] * 10_000
df["potencial"] = df["precio_esperado"] / df["median_house_value"] - 1
conf = df[df["confirmado"]]
prioritarios = conf.nlargest(CAPACIDAD, "potencial")      # los 150 en los que alcanza a invertir el capital
precio_relativo = (df["razon_precio_ingreso"] / mediana_grupo)[df["confirmado"]].median()
ganancia = {f: 1_000_000 * f * prioritarios["potencial"].mean() for f in ESCENARIOS}
print(f"Los confirmados se venden al {100 * precio_relativo:.0f} % del precio de sus comparables")
print(f"Los {CAPACIDAD} elegidos: potencial desde {100 * prioritarios['potencial'].min():.0f} %; "
      f"mediana {100 * prioritarios['potencial'].median():.0f} %")
print({f"f = {f}": miles(g, -3) for f, g in ganancia.items()})
print(f"De los {CAPACIDAD}, siguen confirmados con el 4 %: {len(set(prioritarios.index) & c4)}; "
      f"con el 6 %: {len(set(prioritarios.index) & c6)}")

In [ ]:
# Figura: el paso 4. De los confirmados se eligen los de mayor potencial (el capital alcanza para unos 150).
orden_pot = conf.sort_values("potencial", ascending=False)
techo_fig = 4.0                                   # el eje se corta en 400 %; los que pasan se rotulan
fig, ax = plt.subplots(figsize=(11, 3.8))
x = np.arange(len(orden_pot))
ax.bar(x, 100 * orden_pot["potencial"].clip(upper=techo_fig), width=0.9, linewidth=0,
       color=[PALETA["principal"] if i < CAPACIDAD else PALETA["neutro"] for i in x])
corte_150 = 100 * orden_pot["potencial"].iloc[CAPACIDAD - 1]
ax.axvline(CAPACIDAD - 0.5, color=PALETA["marino"], linestyle="--", linewidth=1.2)
ax.text(CAPACIDAD + 3, 330, f"Puesto {CAPACIDAD}: potencial de {corte_150:.0f} %\nhasta aquí alcanza el capital",
        fontsize=10.5, color=PALETA["marino"], va="top")
n_mas = int((orden_pot["potencial"] > techo_fig).sum())
ax.text(2, 100 * techo_fig + 8, f"{n_mas} distritos pasan de 400 % (recortados)", fontsize=9.5, color=PALETA["secundario"])
ax.set_ylim(0, 100 * techo_fig + 40); ax.set_xlim(-2, len(orden_pot) + 2)
ax.set_ylabel("Potencial (%)"); ax.set_xticks([])
ax.set_xlabel(f"Los {len(conf)} confirmados, ordenados de mayor a menor potencial")
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
titulo(fig, f"Paso 4: de {len(conf)} a {CAPACIDAD}, los que más rendirían frente a sus comparables",
       f"Verde: los {CAPACIDAD} elegidos (el capital del trimestre alcanza para unos {CAPACIDAD}); gris: confirmados que no caben")
fuente(fig, FUENTE_GS)
guardar(fig, "seleccion_150.png")

In [ ]:
# El cálculo paso a paso con el distrito típico de los 150 (el más cercano a la mediana del potencial).
tipico = prioritarios.loc[(prioritarios["potencial"] - prioritarios["potencial"].median()).abs().idxmin()]
r_comp = tipico["precio_esperado"] / (tipico["median_income"] * 10_000)
pasos_calculo = pd.DataFrame({
    "Pregunta": ["1. ¿Cuánto debería valer?", "2. ¿Qué tan por debajo está?", "3. ¿Cuánto gana cada dólar (f = 1/4)?"],
    "Cálculo": [f"{decimal(r_comp, 2)} años × USD {miles(tipico['median_income'] * 10_000)} = USD {miles(tipico['precio_esperado'], -3)}",
                f"{miles(tipico['precio_esperado'], -3)} ÷ {miles(tipico['median_house_value'])} − 1 = {100 * tipico['potencial']:.0f} %",
                f"0,25 × {100 * tipico['potencial']:.0f} % = {decimal(25 * tipico['potencial'], 1)} %"]})
display(pasos_calculo)
print(f"Distrito típico: {ZONAS[tipico['ocean_proximity']].lower()}, ingreso {str(tipico['nivel_ingreso']).lower()}")

In [ ]:
# Regiones: condados de California (U.S. Census Bureau, 2019).
condados = gpd.read_file(DATOS / "cb_2018_us_county_20m.zip")
condados = condados[condados["STATEFP"] == "06"][["NAME", "geometry"]].rename(columns={"NAME": "condado"}).to_crs(3310)
REGIONES = {
    "Bahía de San Francisco": ["Alameda", "Contra Costa", "Santa Clara", "San Francisco", "San Mateo", "Marin", "Sonoma", "Napa", "Solano"],
    "Sur de California": ["Los Angeles", "Orange", "San Diego", "Imperial", "Riverside", "San Bernardino", "Ventura"],
    "Valle Central": ["Kern", "Fresno", "Tulare", "Kings", "Madera", "Merced", "Stanislaus", "San Joaquin", "Sacramento",
                      "Yolo", "Sutter", "Yuba", "Butte", "Glenn", "Colusa", "Tehama", "Shasta"]}
RESTO = "Resto del estado"
puntos = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326").to_crs(3310)
g = gpd.sjoin_nearest(puntos, condados, how="left").drop(columns="index_right")
g = g[~g.index.duplicated()]
g["region"] = g["condado"].map({c: r for r, l in REGIONES.items() for c in l}).fillna(RESTO)
g["prioritario"] = g.index.isin(prioritarios.index)
regiones = (g[g["prioritario"]].groupby("region")
            .agg(distritos=("region", "size"), potencial_mediano=("potencial", "median"),
                 **{f"ganancia_f{int(f * 100)}": ("potencial", lambda s, f=f: 1_000_000 * f * s.mean()) for f in ESCENARIOS})
            .reindex(list(REGIONES) + [RESTO]))
display(regiones.round(2))

In [ ]:
# Cómo se calcula el potencial promedio de una región: el distrito mediano de dos regiones y el promedio.
filas_ej = []
for reg_ in ("Bahía de San Francisco", "Valle Central"):
    d_ = g[g["prioritario"] & (g["region"] == reg_)].sort_values("potencial")
    t_ = d_.iloc[len(d_) // 2]
    ing_ = t_["median_income"] * 10_000
    filas_ej.append({"Región": reg_,
                     "Distrito": f"{t_['condado']}, {ZONAS[t_['ocean_proximity']].lower()}, ingreso {str(t_['nivel_ingreso']).lower()}",
                     "Ingreso": f"USD {miles(ing_, -2)}", "Años que pagan sus comparables": decimal(t_["precio_esperado"] / ing_, 2),
                     "Precio esperado": f"USD {miles(t_['precio_esperado'], -2)}",
                     "Precio actual": f"USD {miles(t_['median_house_value'])}", "Potencial": f"{100 * t_['potencial']:.0f} %",
                     "Promedio de la región": f"{miles(100 * d_['potencial'].sum())} % ÷ {len(d_)} = {100 * d_['potencial'].mean():.0f} %"})
ejemplos_region = pd.DataFrame(filas_ej).set_index("Región")
display(ejemplos_region.T)

In [ ]:
# Figura: el potencial es un techo (el distrito típico como una escalera de precios).
p_hoy, p_esp = tipico["median_house_value"], tipico["precio_esperado"]
niveles = [("Hoy", p_hoy, PALETA["neutro"]),
           ("Si el precio sube\nuna cuarta parte\n(escenario del plan)", p_hoy + 0.25 * (p_esp - p_hoy), PALETA["principal"]),
           ("Si sube\nla mitad", p_hoy + 0.5 * (p_esp - p_hoy), PALETA["principal_claro"]),
           ("Si se pagara como\nsus comparables\n(el techo)", p_esp, "white")]
extremo = prioritarios.loc[prioritarios["potencial"].idxmax()]
fig = plt.figure(figsize=(11, 4.8))
ax = fig.add_axes([0.0, 0.24, 0.58, 0.64])
for i, (rot, v, color) in enumerate(niveles):
    ax.bar(i, v, width=0.62, color=color, edgecolor=PALETA["marino"] if color == "white" else color,
           linestyle="--" if color == "white" else "-", linewidth=1.3)
    ax.text(i, v + p_esp * 0.03, f"USD {miles(v, -2)}", ha="center", fontsize=11.5, fontweight="bold", color=PALETA["texto"])
ax.set_xticks(range(len(niveles)), [n for n, _, _ in niveles], fontsize=10)
ax.set_yticks([]); ax.set_ylim(0, p_esp * 1.15)
for s_ in ("left", "top", "right"): ax.spines[s_].set_visible(False)
tx = fig.add_axes([0.62, 0.2, 0.37, 0.68]); tx.set_axis_off()
tx.text(0, 0.75, "Una advertencia", fontsize=12, fontweight="bold", color=PALETA["acento"], va="top")
tx.text(0, 0.65, f"Un distrito de {g.loc[extremo.name, 'condado'].replace('Los Angeles', 'Los Ángeles')} a USD {miles(extremo['median_house_value'])} "
        f"«valdría»\n{decimal(1 + extremo['potencial'], 1)} veces su precio (potencial de {100 * extremo['potencial']:.0f} %).\n"
        "Una distancia así suele indicar algo que el censo\nno ve: lo establece la debida diligencia.",
        fontsize=10.5, color=PALETA["texto"], va="top", linespacing=1.35)
titulo(fig, f"El potencial es un techo: el distrito típico «valdría» USD {miles(p_esp, -3)}, pero el plan supone "
       f"que llega a {miles(niveles[1][1], -2)}",
       f"Distrito típico de los {CAPACIDAD} (potencial de {100 * tipico['potencial']:.0f} %): precio de la vivienda en cada escenario")
fuente(fig, FUENTE_GS)
guardar(fig, "potencial_techo.png")

In [ ]:
# La distribución completa del potencial (sin recortar): de aquí salen los 150.
fig, ax = plt.subplots(figsize=(9, 3.6))
bins = np.linspace(0, 100 * conf["potencial"].max() + 25, 50)
ax.hist(100 * conf["potencial"], bins=bins, color=PALETA["neutro"])
ax.hist(100 * prioritarios["potencial"], bins=bins, color=PALETA["principal"])
ax.set_xlabel("Potencial (%): cuánto más valdría si se pagara como sus comparables"); ax.set_ylabel("Distritos")
titulo(fig, f"Los {CAPACIDAD} de mayor potencial (verde) entre los {len(conf)} confirmados",
       "Distribución del potencial; los casos extremos piden especial cuidado en la debida diligencia")
fuente(fig, FUENTE_GS)
guardar(fig, "potencial_distribucion.png")

In [ ]:
# Los escenarios por región: de dónde sale cada cifra.
tabla_escenarios = pd.DataFrame({
    "Distritos": regiones["distritos"].astype(int),
    "Potencial promedio": (100 * regiones["ganancia_f25"] / 250_000).round(0).astype(int).astype(str) + " %",
    "Ganancia por USD 1 M si sube 1/4": regiones["ganancia_f25"].map(lambda v: f"USD {miles(v, -3)}"),
    "Si sube 1/2": regiones["ganancia_f50"].map(lambda v: f"USD {miles(v, -3)}")})
display(tabla_escenarios)
# El escenario multiplica a todas las regiones por la misma fracción: cambia cuánto se gana, no el orden. El orden
# cambiaría si el precio subiera más rápido en unas regiones que en otras, y eso los datos no lo miden.

In [ ]:
# Figura: la disyuntiva del comité. Más rendimiento por dólar o más distritos donde invertir.
fig, ax = plt.subplots(figsize=(9, 4.2))
estilo = {"Bahía de San Francisco": (PALETA["principal"], "extremo: más rendimiento por dólar"),
          "Valle Central": (PALETA["principal"], "extremo: más distritos"),
          "Sur de California": (PALETA["marino"], "término medio"),
          RESTO: (PALETA["neutro"], "menos rendimiento y menos distritos que el Sur")}
for reg_, f_ in regiones.iterrows():
    color_, rot_ = estilo[reg_]
    ax.scatter(f_["distritos"], f_["ganancia_f25"], s=160, color=color_, zorder=3)
    abajo = reg_ == RESTO
    ax.annotate(f"{reg_}\n{int(f_['distritos'])} distritos · USD {miles(f_['ganancia_f25'], -3)}\n({rot_})",
                (f_["distritos"], f_["ganancia_f25"]), xytext=(0, -14) if abajo else (12, 0), textcoords="offset points",
                va="top" if abajo else "center", ha="center" if abajo else "left",
                fontsize=10, color=PALETA["texto"], linespacing=1.25)
ax.set_xlim(0, 95); ax.set_ylim(0, regiones["ganancia_f25"].max() * 1.2)
ax.set_xlabel("Distritos disponibles para invertir (de los 150)")
ax.set_ylabel("Ganancia por cada USD 1 M")
ax.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f} mil")
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
titulo(fig, "Más rendimiento por dólar o más distritos: la Bahía y el Valle Central son los extremos; el Sur, el término medio",
       "Las cuatro regiones de los 150 elegidos (escenario prudente)", y=1.08)
fuente(fig, FUENTE_GS)
guardar(fig, "disyuntiva.png")

In [ ]:
# Los 150 que piden una mirada especial antes de comprar (no se excluyen: van primero en la debida diligencia).
# La S3 limpió para el indicador; aquí la pregunta es qué revisar en campo antes de invertir.
criterios = {"Edad de 52 años o más (el censo corta la edad)": prioritarios["housing_median_age"] >= 52,
             "Entre 100 y 300 habitantes (mediana de pocas viviendas)": prioritarios["population"].between(100, 300),
             "Más de 5 personas por hogar (posible hacinamiento)": prioritarios["personas_hogar"] > 5}
excepciones = pd.Series({k: int(m.sum()) for k, m in criterios.items()})
con_excepcion = int(pd.concat(criterios.values(), axis=1).any(axis=1).sum())
e = excepciones.sort_values()
fig, ax = plt.subplots(figsize=(9, 2.9))
ax.barh(e.index, e.values, color=PALETA["acento"], height=0.55)
for n, v in e.items():
    ax.text(v + 0.3, n, str(v), va="center", fontsize=11)
limpiar_ejes(ax)
titulo(fig, f"{con_excepcion} de los {CAPACIDAD} pasaron la limpieza, pero piden una mirada especial antes de comprar",
       "De los 150 elegidos por potencial (paso 4). S3: ¿sirve el dato para el indicador? S4: ¿qué revisar antes de invertir?")
fuente(fig, FUENTE_GS)
guardar(fig, "excepciones.png")

### Punto de control 3 — La ganancia

**¿Por qué el potencial de los elegidos no es lo que va a ganar el fondo?**
Es un techo: supone que el precio alcanza por completo a sus comparables. Una distancia tan grande suele indicar algo
que el censo no ve; por eso se muestran escenarios y se hace debida diligencia antes de comprar.

**El escenario de la mitad da el doble en todas las regiones. ¿Prueba eso que la decisión es robusta?**
No: la fracción multiplica a todas las regiones por igual, así que el orden no puede cambiar. Lo que sí lo cambiaría
es que el precio suba a distinto ritmo en cada región, y eso los datos no lo miden.

> ¿Por qué sumar las diferencias de precio de 150 distritos no es una ganancia del fondo?

## Parte 4 — Del gráfico exploratorio al gráfico principal

El panel de barras del gráfico principal se construye en tres pasos, cada uno sobre el anterior:

1. **Tinta proporcional:** la longitud de la barra es proporcional al valor; las barras empiezan en cero (Wilke, 2019, pp. 207–210; Knaflic, 2015, pp. 50–53).
2. **Desorden y atención:** lo que no aporta compite por la atención; un color para el mensaje y gris para el contexto, con etiquetas directas (Knaflic, 2015, pp. 71–126).
3. **Título de acción:** el título dice la conclusión, incluida la decisión que el comité debe tomar (Knaflic, 2015, p. 141; Wilke, 2019, p. 267).

Luego, el gráfico principal: un **gráfico de decisión**, no de evidencia. El mapa muestra el *dónde* y las barras el
*cuánto*, porque una longitud se percibe con más precisión que un área (Wilke, 2019, p. 216).

In [ ]:
# Paso 1: tinta proporcional (eje recortado frente a eje desde cero).
r = regiones.sort_values("ganancia_f25")
cortas = {"Bahía de San Francisco": "Bahía", "Sur de California": "Sur", "Valle Central": "Valle Central", RESTO: "Resto"}
etiquetas = [cortas[reg] for reg in r.index]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, desde, nombre in ((a1, 250_000, "Eje recortado: exagera las diferencias"), (a2, 0, "Eje desde cero: tinta proporcional")):
    ax.bar(etiquetas, r["ganancia_f25"], color=PALETA["neutro"], width=0.6)
    ax.set_ylim(desde, r["ganancia_f25"].max() * 1.08)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f} mil")
    ax.set_title(nombre, loc="left", fontsize=11.5, color=PALETA["texto"])
bahia, valle = regiones.loc["Bahía de San Francisco", "ganancia_f25"], regiones.loc["Valle Central", "ganancia_f25"]
aparente = (bahia - 250_000) / (valle - 250_000)          # lo que sugiere la altura de las barras recortadas
titulo(fig, f"Paso 1. Con el eje recortado, la Bahía parece rendir {aparente:.0f} veces lo del Valle Central; "
       f"rinde {decimal(bahia / valle, 1)} veces", "Ganancia por cada USD 1 M (escenario prudente), la misma cifra en los dos paneles", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "tinta_proporcional.png")

In [ ]:
# Paso 2: sobre el eje en cero, quitar el desorden y usar un color para el mensaje.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
colores_def = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
a1.barh(etiquetas, r["ganancia_f25"], color=colores_def, edgecolor="black", height=0.6)
a1.grid(True, color="#999999"); a1.spines["top"].set_visible(True); a1.spines["right"].set_visible(True)
a1.legend(a1.patches, etiquetas, fontsize=8.5, loc="lower right"); a1.set_title("Con desorden", loc="left", fontsize=11.5)
a1.xaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f} mil")
resaltar = ("Bahía de San Francisco", "Valle Central")      # las dos opciones del comité
colores = [PALETA["principal"] if reg in resaltar else PALETA["neutro"] for reg in r.index]
a2.barh(etiquetas, r["ganancia_f25"], color=colores, height=0.6)
for i, v in enumerate(r["ganancia_f25"]):
    a2.text(v + 8_000, i, f"USD {miles(v, -3)}", va="center", fontsize=10.5)
a2.set_xticks([]); a2.spines["bottom"].set_visible(False); a2.spines["left"].set_visible(False)
a2.tick_params(axis="y", length=0); a2.set_xlim(0, r["ganancia_f25"].max() * 1.45)
a2.set_title("Sin desorden: un color dice dónde mirar", loc="left", fontsize=11.5)
fig.subplots_adjust(wspace=0.45)
titulo(fig, "Paso 2. Quitar lo que no aporta y usar un color para el mensaje",
       "Mismo dato: ganancia por cada USD 1 M por región. Verde: los extremos de la disyuntiva; gris: contexto", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "desorden.png")

In [ ]:
# Paso 3: título descriptivo frente a título de acción (con la decisión).
TITULO_ACCION = (f"La Bahía rinde el doble por dólar, pero el Valle Central tiene casi "
                 f"{'cuatro' if round(regiones.loc['Valle Central', 'distritos'] / regiones.loc['Bahía de San Francisco', 'distritos']) == 4 else 'más'} "
                 "veces más distritos")
etiquetas_n = [f"{cortas[reg]} ({int(regiones.loc[reg, 'distritos'])} distritos)" for reg in r.index]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, texto in ((a1, "Ganancia por región"), (a2, TITULO_ACCION)):
    ax.barh(etiquetas_n, r["ganancia_f25"], color=colores, height=0.6)
    for i, v in enumerate(r["ganancia_f25"]):
        ax.text(v + 8_000, i, f"USD {miles(v, -3)}", va="center", fontsize=10)
    ax.set_xticks([]); ax.spines["bottom"].set_visible(False); ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, labelsize=9.5); ax.set_xlim(0, r["ganancia_f25"].max() * 1.4)
    ax.set_title("\n".join(textwrap.wrap(texto, 44)), loc="left", fontsize=12, fontweight="bold", color=PALETA["texto"])
a1.text(0, -0.2, "Título descriptivo: el comité tiene que descubrir el mensaje", transform=a1.transAxes, fontsize=10.5, color=PALETA["secundario"])
a2.text(0, -0.2, "Título de acción: dice la conclusión y la decisión", transform=a2.transAxes, fontsize=10.5, color=PALETA["secundario"])
fig.subplots_adjust(wspace=0.75)
guardar(fig, "titulo.png")

In [ ]:
# Gráfico principal: dónde invertir y cuánto se gana (gráfico de decisión).
from matplotlib.colors import LinearSegmentedColormap


def grafico_principal(anotado=False):
    cond = condados.copy()
    cond["region"] = cond["condado"].map({c_: r_ for r_, l in REGIONES.items() for c_ in l}).fillna(RESTO)
    cmap = LinearSegmentedColormap.from_list("verde", ["#E3F1EF", PALETA["principal"]])
    vmin, vmax = 200_000, regiones["ganancia_f25"].max()
    prio = g[g["prioritario"]]
    fig = plt.figure(figsize=(13, 7.6))
    # Franja: de dónde salen los 150.
    ax_f = fig.add_axes([0.0, 0.83, 1.0, 0.09]); ax_f.set_axis_off(); ax_f.set_xlim(0, 1); ax_f.set_ylim(0, 1)
    pasos = [(miles(int(df["candidato"].sum())), "candidatos: el 5 % más barato\npara su ingreso en el estado"),
             (miles(int(df["confirmado"].sum())), "confirmados: siguen baratos\nfrente a su zona y su ingreso"),
             (miles(CAPACIDAD), "los de mayor potencial: en los que\nalcanza a invertir el capital")]
    for i, (cifra, texto) in enumerate(pasos):
        x = 0.01 + i * 0.25
        ax_f.text(x, 0.5, cifra, fontsize=22, fontweight="bold", va="center",
                  color=PALETA["principal"] if i == 2 else PALETA["marino"])
        ax_f.text(x + 0.062, 0.5, texto, fontsize=10.5, va="center", color=PALETA["texto"], linespacing=1.15)
        if i < 2:
            ax_f.annotate("", xy=(x + 0.235, 0.5), xytext=(x + 0.215, 0.5),
                          arrowprops=dict(arrowstyle="-|>", color=PALETA["secundario"], lw=1.3))
    ax_f.text(0.78, 0.5, "De dónde salen los 150\ndel trimestre", fontsize=10.5, va="center",
              color=PALETA["secundario"], style="italic")
    # Mapa: cada región coloreada por la ganancia por USD 1 M; los 150, puntos del mismo tamaño.
    ax_m = fig.add_axes([0.0, 0.03, 0.55, 0.77]); ax_m.set_axis_off()
    cond.plot(ax=ax_m, color=[cmap((regiones.loc[r_, "ganancia_f25"] - vmin) / (vmax - vmin)) for r_ in cond["region"]],
              edgecolor="white", linewidth=0.3)
    cond.dissolve("region").boundary.plot(ax=ax_m, color="white", linewidth=1.4)
    ax_m.scatter(prio.geometry.x, prio.geometry.y, s=9, color=PALETA["marino"], edgecolor="white", linewidth=0.3, zorder=3)
    rotulos = {RESTO: ((-720, 330), (-280, 300)), "Bahía de San Francisco": ((-720, -60), (-190, -10)),
               "Valle Central": ((720, 150), (40, -60)), "Sur de California": ((720, -330), (330, -400))}
    for reg, ((tx, ty), (px_, py_)) in rotulos.items():
        f = regiones.loc[reg]
        ax_m.annotate(f"{reg}\n{int(f['distritos'])} distritos\nUSD {miles(f['ganancia_f25'], -3)} por millón",
                      xy=(px_ * 1000, py_ * 1000), xytext=(tx * 1000, ty * 1000), fontsize=10, ha="center", va="center",
                      color=PALETA["texto"], linespacing=1.2,
                      arrowprops=dict(arrowstyle="-", color=PALETA["secundario"], lw=0.8, shrinkA=18))
    ax_m.set_xlim(-950_000, 960_000); ax_m.set_ylim(-640_000, 470_000)
    cax = fig.add_axes([0.03, 0.06, 0.16, 0.018])
    cax.imshow(np.linspace(0, 1, 100)[None, :], aspect="auto", cmap=cmap); cax.set_yticks([])
    cax.set_xticks([0, 99], [f"USD {miles(vmin, -3)}", f"USD {miles(vmax, -3)}"], fontsize=9.5)
    cax.set_title("Ganancia por cada USD 1 M", fontsize=10, loc="left", color=PALETA["texto"])
    for s_ in cax.spines.values(): s_.set_visible(False)
    fig.text(0.03, 0.13, f"● uno de los {CAPACIDAD} distritos", fontsize=10, color=PALETA["marino"])
    # Panel derecho: la disyuntiva.
    orden = regiones.sort_values("ganancia_f25").index
    col = [PALETA["principal"] if r_ in resaltar else PALETA["neutro"] for r_ in orden]
    for k, (campo, rot, fmt) in enumerate((("ganancia_f25", "Cuánto rinde cada dólar\nganancia por cada USD 1 M",
                                            lambda v: f"USD {miles(v, -3)}"),
                                           ("distritos", "Cuántos distritos hay para invertir\n"
                                            f"de los {CAPACIDAD} del trimestre", lambda v: f"{int(v)}"))):
        ax = fig.add_axes([0.63 + k * 0.2, 0.14, 0.13, 0.58])
        yy = np.arange(len(orden))
        ax.barh(yy, regiones.loc[orden, campo], color=col, height=0.6)
        for yi, v in zip(yy, regiones.loc[orden, campo]):
            ax.text(v + regiones[campo].max() * 0.03, yi, fmt(v), va="center", fontsize=10.5, color=PALETA["texto"])
        ax.set_yticks(yy, [cortas[r_] if r_ != RESTO else "Resto" for r_ in orden] if k == 0 else [""] * len(yy),
                      fontsize=10.5)
        ax.set_xlim(0, regiones[campo].max() * 1.45); ax.set_xticks([])
        for s_ in ("top", "right", "bottom"): ax.spines[s_].set_visible(False)
        ax.spines["left"].set_color(PALETA["secundario"]); ax.tick_params(axis="y", length=0)
        ax.set_title(rot, fontsize=11, loc="left", color=PALETA["texto"], fontweight="bold")
    fig.text(0.57, 0.035, "Decisión del comité: ¿más rendimiento por dólar en la Bahía o más distritos en el Valle\n"
             f"Central? Verde: los extremos. El Sur es el término medio (USD "
             f"{miles(regiones.loc['Sur de California', 'ganancia_f25'], -3)} con {int(regiones.loc['Sur de California', 'distritos'])} distritos).",
             fontsize=10.5, color=PALETA["texto"])
    fig.text(0.01, 1.0, TITULO_ACCION, fontsize=16, fontweight="bold", color=PALETA["texto"])
    fig.text(0.01, 0.955, f"Los {CAPACIDAD} distritos del trimestre: rendirían USD {miles(ganancia[0.25], -3)} por cada millón "
             "si el precio sube una cuarta parte de lo que le falta para igualar a sus comparables",
             fontsize=11.5, color=PALETA["secundario"])
    fig.text(0.01, -0.03, FUENTE_GS + " Límites de condados: U.S. Census Bureau (2019).", fontsize=8.5,
             color=PALETA["secundario"])
    if anotado:  # dónde se aplica cada recomendación
        for n, (x, y) in enumerate(((0.005, 1.035), (0.625, 0.12), (0.955, 0.2), (0.805, 0.69), (0.515, -0.025),
                                    (0.42, 0.45)), 1):
            fig.text(x, y, str(n), fontsize=12, fontweight="bold", color="white", ha="center", va="center",
                     bbox=dict(boxstyle="circle,pad=0.3", facecolor=PALETA["acento"], edgecolor="none"))
    return fig


guardar(grafico_principal(), "grafico_principal.png")
guardar(grafico_principal(anotado=True), "grafico_principal_anotado.png")

**Las seis decisiones de diseño del gráfico principal** (los números de la versión anotada):

1. Título de acción que dice la conclusión y la decisión.
2. Barras que empiezan en cero (tinta proporcional).
3. Un color para las dos opciones del comité y gris para el contexto.
4. Etiquetas directas en lugar de ejes y leyendas.
5. El escenario y la fuente, declarados.
6. El mapa para el *dónde* y las barras para el *cuánto*: una longitud se compara con más precisión que un área.

### Punto de control 4 — El gráfico principal

**El gráfico de barras por zona estaba bien diseñado. ¿Por qué no es el principal?**
El comité se convence de que el hallazgo es real, pero no sabe dónde invertir: es un gráfico de evidencia (respaldo).

**¿Qué decide el comité con este gráfico?**
Cómo repartir el capital entre regiones: más rendimiento por dólar en la Bahía o más distritos en el Valle Central.

> El mapa ya está coloreado por la ganancia; ¿por qué hace falta además el panel con barras?

## Parte 5 — El SCQA y la lista de verificación

SCQA: Situación, Complicación, Pregunta y Respuesta, en el orden que necesita el comité (StrategyPunk, 2026,
pp. 6–7; Knaflic, 2015, p. 172; Wilke, 2019, p. 334). La celda arma el SCQA con las cifras calculadas arriba, para
que el texto nunca diga algo distinto de lo que muestran los datos.

In [ ]:
n_cand, n_zona, n_conf = int(df["candidato"].sum()), int(df["resiste_zona"].sum()), int(df["confirmado"].sum())
b_, v_ = regiones.loc["Bahía de San Francisco"], regiones.loc["Valle Central"]
scqa = {
    "S": f"Golden State invierte donde la vivienda se vende por debajo de lo que el ingreso sostiene. Antes de comprar, el "
         f"capital del trimestre alcanza para unos {CAPACIDAD} distritos, y antes de comprar el equipo revisa cada uno "
         "(debida diligencia).",
    "C": f"El barrido marcó {miles(n_cand)} «gangas», pero 9 de cada 10 están en el interior, donde todo es más barato. "
         "Invertir con la lista tal cual sería comprar donde el mercado ya valora bien.",
    "Q": "¿En qué distritos debe invertir el fondo este trimestre?",
    "A": f"En los {CAPACIDAD} de mayor potencial entre los {n_conf} que siguen baratos frente a su zona e ingreso: rendirían "
         f"USD {miles(ganancia[0.25], -3)} por cada millón si el precio sube una cuarta parte de lo que le falta para igualar "
         f"a sus comparables. La Bahía rinde USD {miles(b_['ganancia_f25'], -3)} por millón y el Valle Central "
         f"USD {miles(v_['ganancia_f25'], -3)}, pero tiene {int(b_['distritos'])} distritos frente a {int(v_['distritos'])}; "
         f"el Sur es el término medio (USD {miles(regiones.loc['Sur de California', 'ganancia_f25'], -3)} con "
         f"{int(regiones.loc['Sur de California', 'distritos'])} distritos).",
}
lista = pd.DataFrame([
    ("Magnitud", f"Con USD 1 M repartido en los {CAPACIDAD}, si el precio sube una cuarta parte de lo que le falta, el fondo "
                 f"gana USD {miles(ganancia[0.25], -3)}", "Sí"),
    ("Mecanismo", "Sabemos que el precio no acompaña al ingreso aun frente a distritos parecidos; por qué, lo establece "
                  "la debida diligencia antes de comprar", "En parte"),
    ("Robustez", f"Con cortes del 4 % y del 6 % los {n_conf} se mantienen; de los {CAPACIDAD}, siguen "
                 f"{len(set(prioritarios.index) & c4)} y {len(set(prioritarios.index) & c6)}", "Sí"),
    ("Alternativas", f"¿La zona? Se controló: salen {n_cand - n_zona}. ¿El ingreso? Se controló: salen {n_zona - n_conf} más. "
                     "Sin controlar (el censo no lo mide): estado de las viviendas, seguridad, planes de la ciudad",
                     "Sí, las medibles"),
    ("Accionabilidad", f"{CAPACIDAD} distritos con región y potencial para repartir el capital; los {con_excepcion} con "
                       "datos que piden cuidado se revisan primero", "Sí"),
], columns=["Pregunta", "Respuesta", "¿Se cumple?"]).set_index("Pregunta")
for k, v in scqa.items():
    print(f"{k}: {v}\n")
display(lista)

## Errores frecuentes

1. **SCQA desde la mirada del analista.** La Situación describe el dataset y la Respuesta dice «el patrón se sostiene».
2. **Llevar el hallazgo sin verificar.** Los 971 candidatos en lugar de los confirmados: dos de cada tres serían un espejismo.
3. **Un gráfico principal que no lleva a decidir.** La dispersión exploratoria, o unas barras que solo prueban que el hallazgo es real.
4. **Decir qué se encontró y no qué se gana.** Sumar diferencias de precio entre distritos como si fueran ganancia.
5. **Un título que dice más de lo verificado.** «307 oportunidades seguras»: la verificación no garantiza la inversión.

## Síntesis

| Qué hicimos | Por qué importa |
|---|---|
| Revisamos qué traían las «gangas» crudas | Las dos mejores eran el piso del censo |
| Comparamos cada candidato con su zona y su ingreso | Dos de cada tres eran solo casas del interior |
| Medimos la oportunidad como ganancia por millón, con escenarios | El comité compara inversiones así; el potencial es un techo, no una promesa |
| Pasamos del gráfico exploratorio al gráfico de decisión | El comité ve dónde invertir, cuánto se gana y qué debe decidir |
| Escribimos el SCQA con las cifras calculadas | El texto no dice más de lo que muestran los datos |

**Próxima sesión:** ¿dónde está y a qué escala hay que mirarlo? Mapas y normalización.

## Transición al taller práctico

Cada equipo hace las mismas cinco tareas con CONECTA S.A., proyecto A (Retención), en el notebook
`sesion_04_practica_proyecto_A.ipynb`. Datos de partida: `datos/conecta_limpio_s3.csv` y las diez UPZ candidatas
que dejó la S3. Todo se hace en Python; en la S7 y la S8 lo construirán en Power BI.

## Referencias

Diez, D. M., Çetinkaya-Rundel, M., y Barr, C. D. (2019). *OpenIntro statistics* (4.ª ed.). OpenIntro. https://www.openintro.org/book/os/

Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.

Pace, R. K., y Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters, 33*(3), 291–297. https://doi.org/10.1016/S0167-7152(96)00140-X

StrategyPunk. (2026, 29 de marzo). *The Minto pyramid principle: How to communicate like a McKinsey consultant (PDF)*. https://www.strategypunk.com/the-minto-pyramid-principle-how-to-communicate-like-a-mckinsey-consultant-pdf/

U.S. Census Bureau. (2019). *2018 cartographic boundary file, current county and equivalent for United States, 1:20,000,000* [Shapefile].

Wilke, C. O. (2019). *Fundamentals of data visualization: A primer on making informative and compelling figures*. O'Reilly Media.